# Notation for Inference Service Math Model

This section explains the notation used for modeling various aspects of an inference service, especially for transformer-based models and hardware characteristics:

- **𝑊** – Total size of weights
- **𝐿** – Number of transformer layers
- **𝐻ₘₒdₑₗ** – Model hidden size (dimension of hidden states)
- **𝐵** – Bytes per element (precision, e.g., 2 for FP16, 1 for FP8)
- **𝐻ₖᵥ** – Number of key-value heads (for Grouped Query Attention, GQA)
- **𝐻_q** – Number of query heads (for GQA)
- **𝐷ₗₐₜₑₙₜ** – Latent dimension (for Mixture-of-Latents Attention, MLA)
- **𝑇** – Context size (sequence length, e.g., 8K tokens)
- **𝐶** – Communication channel bandwidth (e.g., 12 GB/s for PCIe Gen4)
- **𝜏** – Communication channel latency (e.g., 20 μs for PCIe Gen4)
- **𝐷** – VRAM overhead (fraction, e.g., 0.15–0.20)
- **𝑁** – Batch size
- **𝑅** – GPU FLOPS/sec (FP16 Tensor)
- **𝐵𝑊ₘₑₘ** – GPU VRAM bandwidth

These variables are used to calculate memory usage, throughput, latency, and other performance metrics for inference services.

In [15]:
# Notation for Inference Service Math Model
#
# This section explains the notation used for modeling various aspects of an inference service, especially for transformer-based models and hardware characteristics:
#
# - 𝑊 – Total size of weights
# - 𝐿 – Number of transformer layers
# - 𝐻_hidden – Model hidden size (dimension of hidden states)
# - 𝐵 – Bytes per element (precision, e.g., 2 for FP16, 1 for FP8)
# - 𝐻_kv – Number of key-value heads (for Grouped Query Attention, GQA)
# - 𝐻_q – Number of query heads (for GQA)
# - 𝐷_latentₜ – Latent dimension (for Mixture-of-Latents Attention, MLA)
# - 𝑇 – Context size (sequence length, e.g., 8K tokens)
# - 𝐶 – Communication channel bandwidth (e.g., 12 GB/s for PCIe Gen4)
# - 𝜏 – Communication channel latency (e.g., 20 μs for PCIe Gen4)
# - 𝐷 – VRAM overhead (fraction, e.g., 0.15–0.20)
# - 𝑁 – Batch size
# - 𝑅 – GPU FLOPS/sec (FP16 Tensor)
# - 𝐵𝑊_mem – GPU VRAM bandwidth
#
# These variables are used to calculate memory usage, throughput, latency, and other performance metrics for inference services.

In [18]:


class GPU:
    """
    Class representing a GPU and its key attributes for inference modeling, including price.
    All values are normalized to bytes (B) and FLOPS (not GB/s or TFLOPS).
    """
    def __init__(self, name: str, vram_b: float, vram_bw_bps: float, flops: float, price: float):
        self.name = name
        self.vram_b = vram_b
        self.vram_bw_bps = vram_bw_bps
        self.flops = flops
        self.price = price

    @classmethod
    def from_name(cls, name: str) -> 'GPU':
        """Factory method to build GPU from a GPU name."""
        gpus = {
            "NVIDIA RTX PRO 6000": dict(vram_b=96 * 1024**3, vram_bw_bps=1694.5 * 1e9, flops=310 * 1e12, price=8346),
            "NVIDIA H100": dict(vram_b=80 * 1024**3, vram_bw_bps=3350 * 1e9, flops=989 * 1e12, price=29999),
            "NVIDIA A100-80GB": dict(vram_b=80 * 1024**3, vram_bw_bps=2000 * 1e9, flops=312 * 1e12, price=23300),
            "NVIDIA L40": dict(vram_b=48 * 1024**3, vram_bw_bps=864 * 1e9, flops=181 * 1e12, price=11225),
            "NVIDIA H200": dict(vram_b=141 * 1024**3, vram_bw_bps=4800 * 1e9, flops=989 * 1e12, price=30250),
            "AMD Instinct MI300A": dict(vram_b=128 * 1024**3, vram_bw_bps=5300 * 1e9, flops=981 * 1e12, price=29396),
            "AMD Instinct MI300X": dict(vram_b=192 * 1024**3, vram_bw_bps=5300 * 1e9, flops=1307 * 1e12, price=32000),
        }
        if name not in gpus:
            raise ValueError(f"Unknown GPU: {name}")
        return cls(name=name, **gpus[name])

    @classmethod
    def available_gpus(cls) -> list[str]:
        """Return a list of available GPU names."""
        return [
            "NVIDIA RTX PRO 6000",
            "NVIDIA H100",
            "NVIDIA A100-80GB",
            "NVIDIA L40",
            "NVIDIA H200",
            "AMD Instinct MI300A",
            "AMD Instinct MI300X",
        ]

# Example usage:
# gpu = GPU("NVIDIA H100")
# print(gpu.vram_b, gpu.vram_bw_bps, gpu.flops, gpu.price)


In [20]:

class CommLink:
    """
    Class representing a communication link and its key attributes.
    Bandwidth is in bytes/sec, latency is in seconds.
    For attributes with a range, the main accessor returns the average, and there are methods for low and high values.
    """
    _comm_db: dict[str, dict[str, tuple[float, float]]] = {
        "NVLink": {"bw_gbps": (300, 400), "lat_us": (1, 2)},
        "PCIe Gen4": {"bw_gbps": (12, 15), "lat_us": (18, 25)},
        "PCIe Gen5": {"bw_gbps": (25, 30), "lat_us": (10, 15)},
        "Infiniband HDR": {"bw_gbps": (22, 24), "lat_us": (5, 8)},
        "Infiniband NDR": {"bw_gbps": (45, 48), "lat_us": (2, 5)},
        "Ethernet 100G": {"bw_gbps": (10, 11), "lat_us": (10, 20)},
        "Ethernet 25-40G": {"bw_gbps": (2.5, 4), "lat_us": (50, 100)},
        "Cross-DC (WAN)": {"bw_gbps": (0, 1), "lat_us": (10000, 100000)},
    }

    def __init__(self, name: str) -> None:
        if name not in self._comm_db:
            raise ValueError(f"Unknown Comm Link: {name}")
        self._name: str = name
        self._attrs: dict[str, tuple[float, float]] = self._comm_db[name]

    @property
    def name(self) -> str:
        return self._name

    # Bandwidth (bytes/sec)
    @property
    def bandwidth_bps(self) -> float:
        low, high = self._attrs["bw_gbps"]
        return ((low + high) / 2) * 1e9
    def bandwidth_bps_low(self) -> float:
        return self._attrs["bw_gbps"][0] * 1e9

    def bandwidth_bps_high(self) -> float:
        return self._attrs["bw_gbps"][1] * 1e9

    # Latency (seconds)
    @property
    def latency_s(self) -> float:
        low, high = self._attrs["lat_us"]
        return ((low + high) / 2) * 1e-6

    def latency_s_low(self) -> float:
        return self._attrs["lat_us"][0] * 1e-6

    def latency_s_high(self) -> float:
        return self._attrs["lat_us"][1] * 1e-6

    @classmethod
    def available_links(cls) -> list[str]:
        """Return a list of available comm link names."""
        return list(cls._comm_db.keys())

# Example usage:
# link = CommLink("PCIe Gen4")
# print(link.bandwidth_bps, link.bandwidth_bps_low(), link.bandwidth_bps_high())
# print(link.latency_s, link.latency_s_low(), link.latency_s_high())



In [35]:

class ModelAttributes:
    """
    Provides attributes for several FP16 models (e.g., LLaMA-3-70B, GPT-OSS-20B, etc.).
    Returns model parameters and configuration.
    """
    def __init__(self,
                 name: str,
                 W: int,
                 L: int,
                 H_model: int,
                 B: int,
                 H_kv: int | None = None,
                 H_q: int | None = None,
                 D_latent: int | None = None,
                 attention_type: str | None = None,
                 moe: bool = False):
        self.name = name
        self.W = W
        self.L = L
        self.H_model = H_model
        self.B = B
        self.H_kv = H_kv
        self.H_q = H_q
        self.D_latent = D_latent
        self.attention_type = attention_type
        self.moe = moe

    @classmethod
    def from_name(cls, name: str) -> 'ModelAttributes':
        """Factory method to build ModelAttributes from a model name."""
        models = {
            "LLaMA-3.1-70B": dict(W=70_554_470_400, L=80, H_model=8192, B=2, H_kv=8, H_q=64, D_latent=None, attention_type="GQA", moe=False),
            "GPT-OSS-20B": dict(W=20_910_000_000, L=24, H_model=2880, B=2, H_kv=8, H_q=64, D_latent=None, attention_type="GQA", moe=True),
            "DeepSeek-V3": dict(W=671_000_000_000, L=61, H_model=7168, B=2, H_kv=1, H_q=128, D_latent=512, attention_type="MLA", moe=True),
            "Mistral-Small-24B": dict(W=24_000_000_000, L=40, H_model=5120, B=2, H_kv=8, H_q=32, D_latent=None, attention_type="GQA", moe=False),
            "Qwen2.5-72B": dict(W=72_700_000_000, L=80, H_model=8192, B=2, H_kv=8, H_q=64, D_latent=None, attention_type="GQA", moe=False),
            "LLaMA-3.1-405B": dict(W=405_000_000_000, L=126, H_model=16384, B=2, H_kv=8, H_q=128, D_latent=None, attention_type="GQA", moe=False),
            "Mixtral-8x22B": dict(W=141_000_000_000, L=56, H_model=6144, B=2, H_kv=8, H_q=48, D_latent=None, attention_type="GQA", moe=True),
        }
        if name not in models:
            raise ValueError(f"Unknown model: {name}")
        return cls(name=name, **models[name])

    @classmethod
    def available_models(cls) -> list[str]:
        """Return a list of available model names."""
        return [
            "LLaMA-3.1-70B",
            "GPT-OSS-20B",
            "DeepSeek-V3",
            "Mistral-Small-24B",
            "Qwen2.5-72B",
            "LLaMA-3.1-405B",
            "Mixtral-8x22B",
        ]

    def feasible_pp_prefill_configs(self, gpu) -> set[tuple[int, int, int]]:
        """
        Returns a set of tuples (num_ranks, batch_size, context_size) specifying feasible pipeline parallel (PP) prefill configurations
        for this model instance and the given GPU. Each tuple can be used to configure a pair of PP pipelines (prefill and decode) for inference.
        Only includes configs where weights and KV-cache fit in no more than 75% of the GPU's VRAM.
        The returned set is not exhaustive, but covers common practical configurations.
        """
        feasible = set()
        vram_limit = 0.70 * gpu.vram_b  # No more than 70% of VRAM may be used.
        pp_options = [r for r in [1, 2, 4, 8, 16] if self.L % r == 0]
        batch_options = [1, 2, 4, 8, 16, 32, 64, 128]
        context_options = [512, 1024, 2048, 4096, 8192, 16384, 32768]
        for num_ranks in pp_options:
            weights_per_rank = self.W // num_ranks
            for batch_size in batch_options:
                for context_size in context_options:
                    try:
                        kv = self.KV(batch_size, context_size)
                        total_mem = weights_per_rank * self.B + kv
                        if kv > 0 and weights_per_rank > 0 and total_mem <= vram_limit:
                            feasible.add((num_ranks, batch_size, context_size))
                    except Exception:
                        continue
        return feasible

    def KV(self, N: int, T: int) -> int:
        """
        Calculate the KV-cache size (in bytes) for a batch of size N and context size T.
        - For GQA: N * T * L * H_kv * (H_model / H_q) * 2 * B
        - For MHA: N * T * L * H_model * 2 * B
        - For MLA: N * T * L * H_kv * D_latent * 2 * B
        (2 for K and V, B is bytes per element)
        """
        attn_type = self.attention_type
        # Determine required variables and their types for each attention type
        if attn_type == "MLA":
            required = {
                "L": (self.L, int),
                "H_kv": (self.H_kv, int),
                "D_latent": (self.D_latent, int),
                "B": (self.B, int),
                "N": (N, int),
                "T": (T, int)
            }
        elif attn_type == "MHA":
            required = {
                "L": (self.L, int),
                "H_model": (self.H_model, int),
                "B": (self.B, int),
                "N": (N, int),
                "T": (T, int)
            }
        else:  # GQA (default)
            required = {
                "L": (self.L, int),
                "H_kv": (self.H_kv, int),
                "H_q": (self.H_q, float),
                "H_model": (self.H_model, float),
                "B": (self.B, int),
                "N": (N, int),
                "T": (T, int)
            }
        # Check for None and cast in one loop, assign to locals with correct type
        local_vars = {}
        for k, (v, typ) in required.items():
            if v is None:
                raise ValueError(f"Model attribute '{k}' is None; cannot compute KV-cache size.")
            try:
                local_vars[k] = typ(v)
            except Exception as e:
                raise TypeError(f"Model attribute '{k}' could not be cast to {typ.__name__}: {e}")
        # Unpack variables for calculation
        L = local_vars["L"]
        B = local_vars["B"]
        N = local_vars["N"]
        T = local_vars["T"]
        if attn_type == "MLA":
            H_kv = local_vars["H_kv"]
            D_latent = local_vars["D_latent"]
            return int(N * T * L * H_kv * D_latent * 2 * B)
        elif attn_type == "MHA":
            H_model = local_vars["H_model"]
            return int(N * T * L * H_model * 2 * B)
        else:  # GQA
            H_kv = local_vars["H_kv"]
            H_q = local_vars["H_q"]
            H_model = local_vars["H_model"]
            return int(N * T * L * H_kv * (H_model / H_q) * 2 * B)


In [22]:

class PPRank:
    """
    Represents a pipeline parallel rank in a PPInferenceNode.
    Each PPRank uses a single GPU and communicates with the next via a CommLink.
    """
    def __init__(self, gpu: 'GPU', comm_link: 'CommLink', model: ModelAttributes, num_layers: int) -> None:
        self.gpu: GPU = gpu
        self.comm_link: CommLink = comm_link
        self.num_layers: int = num_layers
        self.hidden_size: int = model.H_model
        self.bytes_per_elem: int = model.B
        self.model: ModelAttributes = model

    @property
    def weight_count(self) -> int:
        """
        Calculates the number of weights assigned to this PPRank.
        Proportional to the number of layers in this rank.
        """
        total_weights = self.model.W
        total_layers = self.model.L
        if total_weights is None or total_layers is None or total_layers == 0:
            return 0
        # Distribute weights proportionally to the number of layers in this rank
        return int(total_weights * self.num_layers / total_layers)

    def T_prefill(self, batch_size: int, context_length: int) -> float:
        """
        Corrected Prefill Time with explicit H^2 scaling for projections
        and T^2 scaling for the attention matrix.
        """
        N = batch_size
        T = context_length
        H = self.hidden_size
        L_rank = self.num_layers
        
        # 1. Compute Time (FLOPS)
        # Projections: Q, K, V, and O projections + MLP (up/down/gate)
        # Most models approximate total linear FLOPS as 2 * N * T * W_per_rank
        # But to be explicit about the H^2 in the attention block projections:
        # flops_attn_projections = L_rank * (8 * N * T * H**2) 
        
        # Linear FLOPS (including MLP and Attention Projections)
        flops_linear = 2 * N * T * self.weight_count
        
        # Quadratic Attention FLOPS (The QK^T and Score * V part)
        # Standard formula: 2 * N * L_rank * H * T^2
        flops_quadratic = 2 * N * L_rank * H * (T**2)
        
        total_flops = flops_linear + flops_quadratic
        t_compute = total_flops / self.gpu.flops

        # 2. Memory Time (VRAM BW)
        # Must read weights + write the resulting KV cache
        weight_bytes = self.weight_count * self.bytes_per_elem
        kv_bytes = self.KV(N, T)
        
        vram_util = 0.80
        t_memory = (weight_bytes + kv_bytes) / (self.gpu.vram_bw_bps * vram_util)

        return max(t_compute, t_memory)



    def X_prefill(self, batch_size: int = 1, context_length: int = 1) -> float:
        """
        Calculates the transfer time (sec) for bytes transferred to the next PPRank during prefill.
        Assumes transfer of hidden states for the batch and context.
        """
        bytes_to_transfer: int = batch_size * context_length * self.hidden_size * self.bytes_per_elem
        bandwidth = self.comm_link.bandwidth_bps
        if bandwidth is None or bandwidth == 0:
            raise ValueError("CommLink bandwidth is None or zero.")
        return self.comm_link.latency_s + (bytes_to_transfer / float(bandwidth))

    def KV(self, N: int, T: int) -> int:
        """
        Calculates the KV-cache size (in bytes) for this PPRank, proportional to its number of layers.
        """
        # Calculate the fraction of layers this rank is responsible for
        total_layers = self.model.L
        if total_layers == 0:
            return 0
        # Use the model's KV_req for the full model, then scale by this rank's layer fraction
        full_kv = self.model.KV(N, T)
        return int(full_kv * self.num_layers / total_layers)
    
    def T_decode(self, N: int, T: int) -> float:
        """
        Corrected Decode Time:
        Time = max(Compute_Time, Memory_Access_Time)
        In decoding, we are almost always Memory Bound by (Weights + KV-Cache).
        """
        # 1. Compute time (FLOPS)
        # Using 2 * W as a heuristic for forward pass FLOPS per token
        flops_per_token = (2 * self.weight_count) * N
        t_compute = flops_per_token / self.gpu.flops

        # 2. Memory bandwidth time (The 'Memory Wall')
        # We must read ALL weights + the current KV cache from VRAM
        weights_per_rank = self.weight_count * self.model.B
        kv_cache_bytes = self.KV(N, T)

        vram_bw_utilization_factor = 0.75  # Assume 75% of VRAM bandwidth is usable

        t_memory = (weights_per_rank + kv_cache_bytes) / (self.gpu.vram_bw_bps * vram_bw_utilization_factor)
        
        # Decoding is usually memory-bound, so we take the max
        return max(t_compute, t_memory)

    def X_decode(self, batch_size: int = 1) -> float:
        """
        Calculates the transfer time (sec) for bytes transferred to the next PPRank during decode (T=1).
        Assumes transfer of hidden states for the batch and a single token.
        """
        bytes_to_transfer = batch_size * self.hidden_size * self.bytes_per_elem
        # Add the base latency (tau) to the transfer time
        return self.comm_link.latency_s + (bytes_to_transfer / self.comm_link.bandwidth_bps)



In [23]:

class PPInferenceNode:
    """
    Represents an LLM node configured with pipeline parallelism.
    Consists of PPRanks with equal layer allocation.
    """
    def __init__(self, ranks: list['PPRank']) -> None:
        self.ranks: list[PPRank] = ranks

    @classmethod
    def build_ranks(cls, model: ModelAttributes, gpu_list: list['GPU'], comm_link_name: str) -> list['PPRank']:
        """
        Construct a list of PPRank objects for pipeline parallelism.

        Args:
            model (ModelAttributes): The model configuration object.
            gpu_list (list[GPU]): List of GPU objects, one per pipeline rank.
            comm_link_name (str): Name of the communication link type (e.g., 'PCIe Gen4') between the ranks.

        Returns:
            list[PPRank]: List of PPRank objects, each representing a pipeline stage.  

        Logic:
            - Each PPRank is assigned a GPU and a CommLink.
            - Each rank is assigned an equal number of layers (layers_per_rank).
            - Model-dependent parameters (hidden_size, bytes_per_elem, num_layers) are extracted from the model.
        """
        num_layers = model.L
        num_ranks = len(gpu_list)
        layers_per_rank: int = num_layers // num_ranks
        hidden_size = model.H_model
        bytes_per_elem = model.B

        return [
            PPRank(gpu_list[i], CommLink(comm_link_name), model, layers_per_rank)
            for i in range(num_ranks)
        ]

    def get_rank(self, idx: int) -> PPRank:
        return self.ranks[idx]

    def TTFT_no_chunking(self, N: int, T: int) -> float:
        """
        Compute the Total Time to First Token (TTFT) for the pipeline node.  Assume no chunking (microbatching)
        N: batch size
        T: context length (sequence length)
        Returns TTFT in seconds.
        """
        # Prefill time for each rank (compute + comm), pipeline is sequential for first token
        total = 0.0
        for i, rank in enumerate(self.ranks):
            t_compute = rank.T_prefill(batch_size=N, context_length=T)
            t_comm = 0.0
            if i < len(self.ranks) - 1:
                t_comm = rank.X_prefill(batch_size=N, context_length=T)
            total += t_compute + t_comm
        return total

    def X_kv(self, N: int, T: int, network_link: CommLink, parallel: bool = True) -> float:
        """
        Calculate the time to move the KV cache from the Prefill PPInferenceNode to the Decode PPInferenceNode
        assuming each PP rank in the Prefill node transfers its local KV cache to its corresponding rank in the 
        Decode node. This transfer can occur in parallel or sequentially based on the 'parallel' flag.

        N: batch size
        T: context length
        network_link: The CommLink representing the connection between prefill and decode clusters.
        parallel: 
            If True, assumes each PP rank has a dedicated link (or multi-rail) to transfer 
            data simultaneously. Time is limited by the largest rank + 1 latency.
            
            If False, assumes a single shared link where ranks must send data one after another. 
            Time is the sum of all data / bandwidth + (num_ranks * latency).
        """
        if parallel:
            # Parallel: Max(Rank_KV) / BW + Latency
            # Best for: Multi-rail InfiniBand or dedicated NICs per GPU
            max_kv_bytes_in_a_rank = max(rank.KV(N, T) for rank in self.ranks)
            transfer_time = (max_kv_bytes_in_a_rank / network_link.bandwidth_bps) + network_link.latency_s
        else:
            # Sequential: Sum(All_KV) / BW + (PP * Latency)
            # Best for: Shared single NIC or standard TCP/IP sequential streams
            total_kv_bytes = sum(rank.KV(N, T) for rank in self.ranks)
            transfer_time = (total_kv_bytes / network_link.bandwidth_bps) + (len(self.ranks) * network_link.latency_s)
        
        return transfer_time


    def TTFT_chunking_disaggregated(self, N: int, T: int, M: int, network_link: CommLink, parallel: bool = True) -> float:
        """
        Calculates TTFT including the time to move the KV cache to the decode node.  The assumption is that
        each Prefill node has a separate network_link to its corresponding Decode node, and the KV cache transfer occurs in parallel

        N: batch size
        T: context length (sequence length)
        M: chunk size (microbatch size)
        network_link: Communication link between Prefill and Decode nodes
        parallel: 
            If True, assumes each PP rank has a dedicated link (or multi-rail) to transfer 
            data simultaneously. Time is limited by the largest rank + 1 latency.
            
            If False, assumes a single shared link where ranks must send data one after another. 
            Time is the sum of all data / bandwidth + (num_ranks * latency).
        """
        num_chunks = (T + M - 1) // M
        if num_chunks == 0:
            return 0.0

        # 1. Rank Latencies (Compute + Internal PP Comm)
        rank_latencies = []
        for i, rank in enumerate(self.ranks):
            # Use active weights for compute if MoE
            t_comp = rank.T_prefill(batch_size=N, context_length=M) 
            t_comm = rank.X_prefill(batch_size=N, context_length=M) if i < len(self.ranks) - 1 else 0.0
            rank_latencies.append(t_comp + t_comm)

        # 2. Pipeline Fill (Time for first chunk to reach the end of Rank 3)
        t_fill = sum(rank_latencies)

        # 3. Pipeline Drain (Remaining chunks limited by bottleneck)
        # The bottleneck is the slowest stage (Compute + Comm overhead)
        t_bottleneck = max(rank_latencies)
        t_prefill_total = t_fill + (num_chunks - 1) * t_bottleneck

        # 4. KV Cache Handover (The 'Disaggregation' Penalty)
        # In PD disaggregation, the Decode node cannot start until it receives the KV cache.
        t_kv_transfer = self.X_kv(N, T, network_link, True)

        # Total TTFT is prefill completion + data transfer to the decode cluster
        return t_prefill_total + t_kv_transfer


    def TPOT_throughput(self, N: int, T: int) -> float:
        """
        Calculate the Time Per Output Token (TPOT) for the pipeline node (throughput mode).
        N: batch size
        T: context length (sequence length)
        Returns TPOT in seconds.
        """
        # For each output token, all ranks must process the token and transfer hidden states.
        # The slowest stage (compute or comm) determines the pipeline throughput per token.
        stage_times = []
        for i, rank in enumerate(self.ranks):
            t_compute = rank.T_decode(N, T)
            t_comm = 0.0
            if i < len(self.ranks) - 1:
                t_comm = rank.X_decode(batch_size=N)
            stage_times.append(t_compute + t_comm)
        # TPOT is determined by the slowest stage (pipeline bottleneck)
        return max(stage_times)

    def TPOT_latency(self, N: int, T: int) -> float:
        """
        Calculate the Time Per Output Token (TPOT) for the pipeline node (sequential latency mode).
        N: batch size
        T: context length (sequence length)
        Returns TPOT in seconds.
        """
        # For each output token, all ranks must process the token and transfer hidden states sequentially.
        total_time = 0.0
        for i, rank in enumerate(self.ranks):
            t_compute = rank.T_decode(N, T)
            t_comm = 0.0
            if i < len(self.ranks) - 1:
                t_comm = rank.X_decode(batch_size=N)
            total_time += t_compute + t_comm
        return total_time



# Example: Instantiate PPInferenceNode for LLaMA-3.1-70B with pipeline parallelism (PP=4)
#
# from your_module import GPU, CommLink, ModelAttributes, PPInferenceNode
#
# model = ModelAttributes.from_name("LLaMA-3.1-70B")
#
# # For the Prefill pipeline, use four GPUs and Infiniband HDR for inter-rank communication
# gpu_list = [GPU("NVIDIA RTX PRO 6000") for _ in range(4)]
# prefill_ranks = PPInferenceNode.build_ranks(model, gpu_list, "Infiniband HDR")
# prefill_node = PPInferenceNode(prefill_ranks)
#
# # Access the first rank:
# rank0 = prefill_node.get_rank(0)
# print(rank0.num_layers, rank0.gpu.name, rank0.comm_link.name)
#
# # For the Decode pipeline, use four GPUs and Infiniband HDR for inter-rank communication
# decode_gpu_list = [GPU("NVIDIA RTX PRO 6000") for _ in range(4)]
# decode_ranks = PPInferenceNode.build_ranks(model, decode_gpu_list, "Infiniband HDR")
# decode_node = PPInferenceNode(decode_ranks)
#
# # Example: Compute KV transfer time from prefill to decode ranks via PCIe Gen4
# pcie_link = CommLink("PCIe Gen4")
# kv_transfer_time = prefill_node.KV_transfer_time_for_PD_disaggregation(N=4, T=8192, network_link=pcie_link, parallel=True)
# print(f"KV transfer time (PCIe Gen4, parallel): {kv_transfer_time:.4f} sec")
#
# # Compute TTFT for batch size 4, context 8192
# ttft = prefill_node.TTFT_no_chunking(N=4, T=8192)
# print(f"TTFT: {ttft:.4f} sec")
#
# # Compute TPOT (throughput and latency modes)
# tpot_throughput = prefill_node.TPOT_throughput(N=4, T=8192)
# print(f"TPOT (throughput): {tpot_throughput:.6f} sec")
# tpot_latency = prefill_node.TPOT_latency(N=4, T=8192)
# print(f"TPOT (latency): {tpot_latency:.6f} sec")


In [38]:
# --- Example Usage ---
if __name__ == "__main__":
    model = ModelAttributes.from_name("LLaMA-3.1-70B")
    gpu = GPU.from_name("NVIDIA RTX PRO 6000")
    configs = model.feasible_pp_prefill_configs(gpu)
    # print(f"Feasible PP prefill pipeline configurations for LLaMA-3.1-70B on {gpu.name} (<=75% VRAM):")
    # for cfg in sorted(configs):
    #     print(f"  num_ranks={cfg[0]}, batch_size={cfg[1]}, context_size={cfg[2]}")

In [41]:
import pandas as pd

model = ModelAttributes.from_name("LLaMA-3.1-70B")
gpu = GPU.from_name("NVIDIA H100")
configs = model.feasible_pp_prefill_configs(gpu)

batch_sizes = sorted(set(cfg[1] for cfg in configs))
context_sizes = sorted(set(cfg[2] for cfg in configs))

table = pd.DataFrame('', index=batch_sizes, columns=context_sizes)
for num_ranks, batch_size, context_size in configs:
    current = table.at[batch_size, context_size]
    if pd.isna(current) or not str(current).strip():
        table.at[batch_size, context_size] = str(num_ranks)
    else:
        current_set = set(map(int, str(current).split(',')))
        current_set.add(num_ranks)
        table.at[batch_size, context_size] = ','.join(map(str, sorted(current_set)))
        
table.index.name = 'Batch Size'
table.columns.name = 'Context Size'
table

Context Size,512,1024,2048,4096,8192,16384,32768
Batch Size,,,,,,,
1,"4,8,16","4,8,16","4,8,16","4,8,16","4,8,16","4,8,16","4,8,16"
2,"4,8,16","4,8,16","4,8,16","4,8,16","4,8,16","4,8,16","4,8,16"
4,"4,8,16","4,8,16","4,8,16","4,8,16","4,8,16","4,8,16",16
8,"4,8,16","4,8,16","4,8,16","4,8,16","4,8,16",16,
16,"4,8,16","4,8,16","4,8,16","4,8,16",16,,
32,"4,8,16","4,8,16","4,8,16",16,,,
64,"4,8,16","4,8,16",16,,,,
128,"4,8,16",16,,,,,
